In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("techsash/waste-classification-data")

print("Path to dataset files:", path)

In [ ]:
!pip install tensorflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import skimage.io
import tensorflow

import os
import random
from PIL import Image

from tqdm import tqdm

from skimage.io import imread, imshow
from skimage.transform import resize

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.vgg16 import VGG16

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import InputLayer, Dense, Flatten, BatchNormalization, Dropout, Activation
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.preprocessing.image import load_img, img_to_array

import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

In [ ]:
# Path setup
base_path = os.path.join(path, 'DATASET', 'TRAIN')
categories = ['O', 'R']

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(12, 6))

for row, category in enumerate(categories):
    category_path = os.path.join(base_path, category)
    image_files = os.listdir(category_path)
    selected_images = random.sample(image_files, 3)

    for col, img_file in enumerate(selected_images):
        img_path = os.path.join(category_path, img_file)
        img = Image.open(img_path)

        ax = axes[row][col]
        ax.imshow(img)
        ax.set_title(category)
        ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
train_dir = os.path.join(path, 'DATASET', 'TRAIN')

labels = ['O', 'R']
counts = [len(os.listdir(os.path.join(train_dir, label))) for label in labels]

plt.pie(counts, labels=labels, autopct='%1.1f%%')
plt.title("TRAIN Data Class Distribution")
plt.axis('equal')
plt.show()

In [ ]:
train_datagen = ImageDataGenerator(rescale = 1.0 / 255.0,
                                   zoom_range = 0.4,
                                   rotation_range = 10,
                                   horizontal_flip = True,
                                   vertical_flip = True,
                                   validation_split = 0.2)

valid_datagen = ImageDataGenerator(rescale = 1.0 / 255.0,
                                   validation_split = 0.2)

test_datagen  = ImageDataGenerator(rescale = 1.0 / 255.0)

In [ ]:
train_dataset  = train_datagen.flow_from_directory(directory = os.path.join(path, 'DATASET', 'TRAIN'),
                                                   target_size = (224,224),
                                                   class_mode = 'binary',
                                                   batch_size = 128,
                                                   subset = 'training')



valid_dataset = valid_datagen.flow_from_directory(directory = os.path.join(path, 'DATASET', 'TRAIN'),
                                                  target_size = (224,224),
                                                  class_mode = 'binary',
                                                  batch_size = 128,
                                                  subset = 'validation')

In [ ]:
# Defining Model

base_model = VGG16(input_shape=(224,224,3),
                   include_top=False,
                   weights="imagenet")

In [ ]:
# Freezing Layers

for layer in base_model.layers:
    layer.trainable=False

In [ ]:
# Summary

base_model.summary()

In [ ]:
# Defining Layers

model=Sequential()
model.add(base_model)
model.add(Dropout(0.2))
model.add(Flatten())
model.add(BatchNormalization())
model.add(Dense(512,kernel_initializer='he_uniform'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(512,kernel_initializer='he_uniform'))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
# Summary

model.summary()

In [ ]:
# Model Compile

adam = tensorflow.keras.optimizers.Adam(learning_rate=0.001)

model.compile(loss='binary_crossentropy',
              metrics=['accuracy'],
              optimizer=adam)

In [ ]:
# Defining Callbacks

filepath = './best_model.keras'

earlystopping = EarlyStopping(monitor='val_accuracy',
                              mode='max',
                              patience=2,
                              verbose=1)

checkpoint = ModelCheckpoint(filepath,
                             monitor='val_accuracy',
                             mode='max',
                             save_best_only=True,
                             verbose=1)

callback_list = [earlystopping, checkpoint]

In [ ]:
# Model Fitting

model_history = model.fit(train_dataset,
                          validation_data=valid_dataset,
                          epochs=5,
                          callbacks=callback_list,
                          verbose=1)

In [ ]:
# Summarize the model loss

plt.plot(model_history.history['loss'])
plt.plot(model_history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()


# Summarize model accuracy

plt.plot(model_history.history['accuracy'])
plt.plot(model_history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

In [ ]:
# Test Data

test_data = test_datagen.flow_from_directory(directory = os.path.join(path, 'DATASET', 'TEST'),
                                             target_size = (224,224),
                                             class_mode = 'binary',
                                             batch_size = 128)

In [ ]:

model.evaluate(test_data)

In [ ]:
# Load and preprocess the image
img = load_img(os.path.join(path, 'DATASET', 'TEST', 'O', 'O_12572.jpg'), target_size=(224, 224))
img_array = img_to_array(img) / 255.0  # Normalize to [0,1]
plt.imshow(img_array)
plt.axis('off')
plt.show()


# Expand dimensions to match model input shape: (1, 224, 224, 3)
img_array = np.expand_dims(img_array, axis=0)

# Predict
prediction = model.predict(img_array)

# Interpret the prediction
if prediction[0][0] > 0.5:
    print("The image belongs to **Recycle waste** category")
else:
    print("The image belongs to **Organic waste** category")

In [ ]:
model.save('model.h5')

In [ ]:
from google.colab import files
import io

# Trigger file upload
uploaded = files.upload()

for filename in uploaded.keys():
    # Load the uploaded image
    img = load_img(io.BytesIO(uploaded[filename]), target_size=(224, 224))
    img_array = img_to_array(img) / 255.0  # Normalize

    # Display the image
    plt.imshow(img_array)
    plt.title(f"Uploaded: {filename}")
    plt.axis('off')
    plt.show()

    # Expand dimensions for prediction
    img_batch = np.expand_dims(img_array, axis=0)

    # Predict
    prediction = model.predict(img_batch)

    # Display result
    print(f"Prediction raw score: {prediction[0][0]:.4f}")
    if prediction[0][0] > 0.5:
        print(f"The image '{filename}' belongs to **Recycle waste** category.")
    else:
        print(f"The image '{filename}' belongs to **Organic waste** category.")

In [ ]:
from google.colab import files

# Download the native Keras version (recommended)
print("Downloading best_model.keras...")
files.download('best_model.keras')

In [ ]:
!pip install -q gradio

In [ ]:
import os, time
import numpy as np
import tensorflow as tf
import gradio as gr
from PIL import Image
from tensorflow.keras.models import load_model

# ---------- Model ----------
MODEL_PATH = "best_model.keras"
IMG_SIZE = (224, 224)
LOW_CONF = 0.70  # below this, show a "retake photo" warning

if os.path.exists(MODEL_PATH):
    pred_model = load_model(MODEL_PATH)
else:
    pred_model = globals().get("model")  # fallback to model trained in this notebook
    if pred_model is None:
        raise FileNotFoundError("best_model.keras not found and no `model` in memory.")

CATEGORIES = {
    "recyclable": dict(
        emoji="♻️", title="Recyclable Waste", bin="Blue bin", color="#3b82f6",
        tips=["Rinse off food or liquid residue", "Keep it dry and uncontaminated", "Flatten boxes and bottles to save space"],
    ),
    "organic": dict(
        emoji="🍎", title="Organic Waste", bin="Green bin", color="#10b981",
        tips=["Remove plastic wrapping or stickers", "Ideal for composting", "Don't mix with recyclables"],
    ),
}

# ---------- Helpers ----------
def preprocess(image):
    img = np.asarray(image)
    if img.ndim == 2:                      # grayscale -> RGB
        img = np.stack([img] * 3, axis=-1)
    img = img[..., :3]                     # drop alpha channel if present
    img = tf.image.resize(img, IMG_SIZE).numpy().astype("float32") / 255.0
    return np.expand_dims(img, axis=0)

def placeholder_card(msg="Upload or capture an image to get started."):
    return f"""<div class="card empty"><div class="big">🗑️</div><p>{msg}</p></div>"""

def result_card(key, conf, secs):
    c = CATEGORIES[key]
    warn = ""
    if conf < LOW_CONF:
        warn = '<div class="warn">⚠️ Low confidence. Retake with a single item, plain background and good lighting.</div>'
    tips = "".join(f"<li>{t}</li>" for t in c["tips"])
    return f"""
    <div class="card" style="--accent:{c['color']}">
        <div class="row">
            <div class="big">{c['emoji']}</div>
            <div>
                <span class="badge">{c['bin']}</span>
                <h2>{c['title']}</h2>
            </div>
        </div>
        <div class="meta"><span>Confidence</span><b>{conf:.1%}</b></div>
        <div class="bar"><div style="width:{conf*100:.1f}%"></div></div>
        {warn}
        <h4>Disposal tips</h4>
        <ul>{tips}</ul>
        <div class="foot">Inference time: {secs*1000:.0f} ms</div>
    </div>"""

def stats_html(state):
    total = state["recyclable"] + state["organic"]
    return f"""
    <div class="chips">
        <div class="chip"><b>{total}</b><span>Scanned</span></div>
        <div class="chip" style="--accent:#3b82f6"><b>{state['recyclable']}</b><span>♻️ Recyclable</span></div>
        <div class="chip" style="--accent:#10b981"><b>{state['organic']}</b><span>🍎 Organic</span></div>
    </div>"""

# ---------- Core ----------
def classify(image, state):
    if image is None:
        return placeholder_card("⚠️ No image received. Upload or capture one first."), state["hist"], stats_html(state), state
    try:
        t0 = time.time()
        p = float(pred_model.predict(preprocess(image), verbose=0)[0][0])
        secs = time.time() - t0

        key = "recyclable" if p > 0.5 else "organic"
        conf = p if key == "recyclable" else 1 - p

        state[key] += 1
        thumb = Image.fromarray(np.asarray(image).astype("uint8")[..., :3])
        thumb.thumbnail((220, 220))
        state["hist"].insert(0, (thumb, f"{CATEGORIES[key]['emoji']} {key.title()} · {conf:.0%}"))
        state["hist"] = state["hist"][:8]

        return result_card(key, conf, secs), state["hist"], stats_html(state), state
    except Exception as e:
        return placeholder_card(f"❌ Prediction error: {e}"), state["hist"], stats_html(state), state

def auto_classify(image, auto, state):
    if not auto or image is None:
        return gr.update(), gr.update(), gr.update(), state
    return classify(image, state)

def clear_all(state):
    return None, placeholder_card()

def new_state():
    return {"recyclable": 0, "organic": 0, "hist": []}

# ---------- Styling ----------
custom_css = """
.gradio-container {max-width: 1100px !important; margin: auto !important;}
.hero {text-align:center; padding:30px 16px; border-radius:18px; color:#fff;
       background:linear-gradient(135deg,#10b981,#3b82f6); box-shadow:0 10px 25px -8px rgba(59,130,246,.5);}
.hero h1 {margin:0; font-size:2.3rem; font-weight:800; color:#fff;}
.hero p {margin:8px 0 0; font-size:1.05rem; opacity:.95;}
.card {border:1px solid rgba(128,128,128,.3); border-left:6px solid var(--accent,#9ca3af);
       border-radius:14px; padding:20px; background:rgba(128,128,128,.07);}
.card.empty {text-align:center; padding:48px 20px; opacity:.8; --accent:#9ca3af;}
.card .row {display:flex; align-items:center; gap:16px;}
.card .big {font-size:3rem; line-height:1;}
.card h2 {margin:4px 0 0; font-size:1.5rem; font-weight:800;}
.card h4 {margin:16px 0 4px;}
.card ul {margin:0; padding-left:20px; line-height:1.7;}
.badge {display:inline-block; padding:3px 12px; border-radius:999px; background:var(--accent);
        color:#fff; font-weight:700; font-size:.8rem; letter-spacing:.03em;}
.meta {display:flex; justify-content:space-between; margin:16px 0 6px; font-size:.95rem;}
.bar {height:12px; border-radius:999px; background:rgba(128,128,128,.25); overflow:hidden;}
.bar > div {height:100%; background:var(--accent); border-radius:999px; transition:width .6s ease;}
.warn {margin-top:14px; padding:10px 14px; border-radius:10px; background:rgba(245,158,11,.15);
       border:1px solid rgba(245,158,11,.5); font-size:.92rem;}
.foot {margin-top:14px; font-size:.8rem; opacity:.6; text-align:right;}
.chips {display:flex; gap:12px;}
.chip {flex:1; text-align:center; padding:12px; border-radius:12px; border:1px solid rgba(128,128,128,.3);
       border-bottom:4px solid var(--accent,#9ca3af); background:rgba(128,128,128,.07);}
.chip b {display:block; font-size:1.6rem;}
.chip span {font-size:.8rem; opacity:.8;}
.action-btn {background:linear-gradient(135deg,#10b981,#3b82f6) !important; color:#fff !important;
             font-weight:700 !important; border:none !important; border-radius:10px !important;
             transition:transform .2s ease, box-shadow .2s ease !important;}
.action-btn:hover {transform:translateY(-2px); box-shadow:0 10px 18px -6px rgba(59,130,246,.55) !important;}
footer {display:none !important;}
"""

theme = gr.themes.Soft(
    primary_hue="emerald", secondary_hue="blue", neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "system-ui", "sans-serif"],
)

# ---------- UI ----------
with gr.Blocks(theme=theme, css=custom_css, title="BinGENIUS Smart Classifier") as styled_app:
    state = gr.State(new_state())

    gr.HTML("""<div class="hero"><h1>🗑️ BinGENIUS Smart Classifier</h1>
               <p>Sort waste with precision. Upload, paste or snap a photo and get the right bin instantly.</p></div>""")

    with gr.Row(equal_height=False):
        with gr.Column(scale=1):
            input_img = gr.Image(
                sources=["upload", "webcam", "clipboard"], type="numpy",
                label="Select / take image", height=380,
            )
            auto = gr.Checkbox(value=True, label="Auto-classify when an image is added")
            with gr.Row():
                submit_btn = gr.Button("Identify Target Bin 🔍", elem_classes="action-btn", scale=3)
                clear_btn = gr.Button("Clear", variant="secondary", scale=1)

        with gr.Column(scale=1):
            result = gr.HTML(placeholder_card())

    gr.Markdown("### 📊 Session stats")
    stats = gr.HTML(stats_html(new_state()))
    history = gr.Gallery(label="Recent scans", columns=4, height=210, object_fit="cover", interactive=False)

    gr.Markdown(
        "<center><small>Binary CNN classifier (recyclable vs organic). Results are a guide, "
        "so check your local waste rules for edge cases.</small></center>"
    )

    outs = [result, history, stats, state]
    submit_btn.click(classify, [input_img, state], outs)
    input_img.change(auto_classify, [input_img, auto, state], outs)
    clear_btn.click(clear_all, state, [input_img, result])

styled_app.launch(share=True, debug=True)

In [ ]:
import os, time
import numpy as np
import tensorflow as tf
import gradio as gr
from PIL import Image
from tensorflow.keras.models import load_model

# ---------- Model ----------
MODEL_PATH = "best_model.keras"
IMG_SIZE = (224, 224)
LOW_CONF = 0.70  # below this, show a "retake photo" warning

if os.path.exists(MODEL_PATH):
    pred_model = load_model(MODEL_PATH)
else:
    pred_model = globals().get("model")  # fallback to model trained in this notebook
    if pred_model is None:
        raise FileNotFoundError("best_model.keras not found and no `model` in memory.")

CATEGORIES = {
    "recyclable": dict(
        emoji="♻️", title="Recyclable Waste", bin="Blue bin", color="#3b82f6",
        tips=["Rinse off food or liquid residue", "Keep it dry and uncontaminated", "Flatten boxes and bottles to save space"],
    ),
    "organic": dict(
        emoji="🍎", title="Organic Waste", bin="Green bin", color="#10b981",
        tips=["Remove plastic wrapping or stickers", "Ideal for composting", "Don't mix with recyclables"],
    ),
}

# ---------- Helpers ----------
def preprocess(image):
    img = np.asarray(image)
    if img.ndim == 2:                      # grayscale -> RGB
        img = np.stack([img] * 3, axis=-1)
    img = img[..., :3]                     # drop alpha channel if present
    img = tf.image.resize(img, IMG_SIZE).numpy().astype("float32") / 255.0
    return np.expand_dims(img, axis=0)

def placeholder_card(msg="Upload or capture an image to get started."):
    return f"""<div class="card empty"><div class="big">🗑️</div><p>{msg}</p></div>"""

def result_card(key, conf, secs):
    c = CATEGORIES[key]
    warn = ""
    if conf < LOW_CONF:
        warn = '<div class="warn">⚠️ Low confidence. Retake with a single item, plain background and good lighting.</div>'
    tips = "".join(f"<li>{t}</li>" for t in c["tips"])
    return f"""
    <div class="card" style="--accent:{c['color']}">
        <div class="row">
            <div class="big">{c['emoji']}</div>
            <div>
                <span class="badge">{c['bin']}</span>
                <h2>{c['title']}</h2>
            </div>
        </div>
        <div class="meta"><span>Confidence</span><b>{conf:.1%}</b></div>
        <div class="bar"><div style="width:{conf*100:.1f}%"></div></div>
        {warn}
        <h4>Disposal tips</h4>
        <ul>{tips}</ul>
        <div class="foot">Inference time: {secs*1000:.0f} ms</div>
    </div>"""

def stats_html(state):
    total = state["recyclable"] + state["organic"]
    return f"""
    <div class="chips">
        <div class="chip"><b>{total}</b><span>Scanned</span></div>
        <div class="chip" style="--accent:#3b82f6"><b>{state['recyclable']}</b><span>♻️ Recyclable</span></div>
        <div class="chip" style="--accent:#10b981"><b>{state['organic']}</b><span>🍎 Organic</span></div>
    </div>"""

# ---------- Core ----------
def classify(image, state):
    if image is None:
        return placeholder_card("⚠️ No image received. Upload or capture one first."), None, state["hist"], stats_html(state), state
    try:
        t0 = time.time()
        p = float(pred_model.predict(preprocess(image), verbose=0)[0][0])
        secs = time.time() - t0

        key = "recyclable" if p > 0.5 else "organic"
        conf = p if key == "recyclable" else 1 - p

        state[key] += 1
        thumb = Image.fromarray(np.asarray(image).astype("uint8")[..., :3])
        thumb.thumbnail((220, 220))
        state["hist"].insert(0, (thumb, f"{CATEGORIES[key]['emoji']} {key.title()} · {conf:.0%}"))
        state["hist"] = state["hist"][:8]

        label = {"Recyclable": p, "Organic": 1 - p}
        return result_card(key, conf, secs), label, state["hist"], stats_html(state), state
    except Exception as e:
        return placeholder_card(f"❌ Prediction error: {e}"), None, state["hist"], stats_html(state), state

def auto_classify(image, auto, state):
    if not auto or image is None:
        return gr.update(), gr.update(), gr.update(), gr.update(), state
    return classify(image, state)

def clear_all(state):
    return None, placeholder_card(), None

def new_state():
    return {"recyclable": 0, "organic": 0, "hist": []}

# ---------- Styling ----------
custom_css = """
.gradio-container {max-width: 1100px !important; margin: auto !important;}
.hero {text-align:center; padding:30px 16px; border-radius:18px; color:#fff;
       background:linear-gradient(135deg,#10b981,#3b82f6); box-shadow:0 10px 25px -8px rgba(59,130,246,.5);}
.hero h1 {margin:0; font-size:2.3rem; font-weight:800; color:#fff;}
.hero p {margin:8px 0 0; font-size:1.05rem; opacity:.95;}
.card {border:1px solid rgba(128,128,128,.3); border-left:6px solid var(--accent,#9ca3af);
       border-radius:14px; padding:20px; background:rgba(128,128,128,.07);}
.card.empty {text-align:center; padding:48px 20px; opacity:.8; --accent:#9ca3af;}
.card .row {display:flex; align-items:center; gap:16px;}
.card .big {font-size:3rem; line-height:1;}
.card h2 {margin:4px 0 0; font-size:1.5rem; font-weight:800;}
.card h4 {margin:16px 0 4px;}
.card ul {margin:0; padding-left:20px; line-height:1.7;}
.badge {display:inline-block; padding:3px 12px; border-radius:999px; background:var(--accent);
        color:#fff; font-weight:700; font-size:.8rem; letter-spacing:.03em;}
.meta {display:flex; justify-content:space-between; margin:16px 0 6px; font-size:.95rem;}
.bar {height:12px; border-radius:999px; background:rgba(128,128,128,.25); overflow:hidden;}
.bar > div {height:100%; background:var(--accent); border-radius:999px; transition:width .6s ease;}
.warn {margin-top:14px; padding:10px 14px; border-radius:10px; background:rgba(245,158,11,.15);
       border:1px solid rgba(245,158,11,.5); font-size:.92rem;}
.foot {margin-top:14px; font-size:.8rem; opacity:.6; text-align:right;}
.chips {display:flex; gap:12px;}
.chip {flex:1; text-align:center; padding:12px; border-radius:12px; border:1px solid rgba(128,128,128,.3);
       border-bottom:4px solid var(--accent,#9ca3af); background:rgba(128,128,128,.07);}
.chip b {display:block; font-size:1.6rem;}
.chip span {font-size:.8rem; opacity:.8;}
.action-btn {background:linear-gradient(135deg,#10b981,#3b82f6) !important; color:#fff !important;
             font-weight:700 !important; border:none !important; border-radius:10px !important;
             transition:transform .2s ease, box-shadow .2s ease !important;}
.action-btn:hover {transform:translateY(-2px); box-shadow:0 10px 18px -6px rgba(59,130,246,.55) !important;}
footer {display:none !important;}
"""

theme = gr.themes.Soft(
    primary_hue="emerald", secondary_hue="blue", neutral_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "system-ui", "sans-serif"],
)

# ---------- UI ----------
with gr.Blocks(theme=theme, css=custom_css, title="BinGENIUS Smart Classifier") as styled_app:
    state = gr.State(new_state())

    gr.HTML("""<div class="hero"><h1>🗑️ BinGENIUS Smart Classifier</h1>
               <p>Sort waste with precision. Upload, paste or snap a photo and get the right bin instantly.</p></div>""")

    with gr.Row(equal_height=False):
        with gr.Column(scale=1):
            input_img = gr.Image(
                sources=["upload", "webcam", "clipboard"], type="numpy",
                label="Select / take image", height=380,
            )
            auto = gr.Checkbox(value=True, label="Auto-classify when an image is added")
            with gr.Row():
                submit_btn = gr.Button("Identify Target Bin 🔍", elem_classes="action-btn", scale=3)
                clear_btn = gr.Button("Clear", variant="secondary", scale=1)

        with gr.Column(scale=1):
            result = gr.HTML(placeholder_card())
            probs = gr.Label(label="Class probabilities", num_top_classes=2)

    gr.Markdown("### 📊 Session stats")
    stats = gr.HTML(stats_html(new_state()))
    history = gr.Gallery(label="Recent scans", columns=4, height=210, object_fit="cover", interactive=False)

    gr.Markdown(
        "<center><small>Binary CNN classifier (recyclable vs organic). Results are a guide, "
        "so check your local waste rules for edge cases.</small></center>"
    )

    outs = [result, probs, history, stats, state]
    submit_btn.click(classify, [input_img, state], outs)
    input_img.change(auto_classify, [input_img, auto, state], outs)
    clear_btn.click(clear_all, state, [input_img, result, probs])

styled_app.launch(share=True, debug=True)

```markdown
### 🚀 How to deploy this permanently on Hugging Face Spaces:

To host this application permanently and for free on Hugging Face:
1. Create a free account on [Hugging Face](https://huggingface.co/).
2. Click on **Spaces** -> **Create new Space**.
3. Choose **Gradio** as the SDK, name your space, and select public or private.
4. Once created, upload your model file (`best_model.keras` or `model.h5`) to your space.
5. Create an `app.py` file in the space and paste the Gradio application code shown below:

```python
import gradio as gr
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.models import load_model

# Load your trained model
model = load_model('best_model.keras')

def predict_waste(image):
    if image is None:
         return "Please upload an image or capture one."
    img = tf.image.resize(image, (224, 224)) / 255.0
    img_array = img_to_array(img)
    img_batch = np.expand_dims(img_array, axis=0)
    prediction = model.predict(img_batch)[0][0]
    
    if prediction > 0.5:
        return "♻️ RECYCLABLE WASTE\n\nPlace this in the Blue/Recycling Dustbin! 🔵"
    else:
        return "🍎 ORGANIC WASTE\n\nPlace this in the Green/Organic Dustbin! 🟢"

interface = gr.Interface(
    fn=predict_waste,
    inputs=gr.Image(sources=["upload", "webcam"], type="numpy"),
    outputs=gr.Textbox(label="Correct Dustbin Destination"),
    title="🗑️ AI Waste Classifier",
    theme="soft"
)

interface.launch()
```
6. Create a `requirements.txt` file in your Hugging Face space containing:
```text
tensorflow
keras
numpy
pillow
```
Your app will automatically build and stay online indefinitely!
```